In [ ]:
from pkg.forecast.input_preparation import prepare_graphcast_input
from pkg.forecast.run_forecast import run_forecast
from pkg.gcs_utils.client import download_file

from datetime import datetime, timedelta
import cdsapi

# Download and Forecast 18-02-2022 22:00 UTC

## Download Data for GraphCast

Downloads data from the ERA5 database and prepares them to be suitable input for GraphCast.
As parameters the desired date and times can be chosen.
Addiotionally can be chosen between 13 and 37 pressure levels and a resolution of 0.25 or 1.0 degrees to match different GraphCast models.

Data is saved locally and uploaded to a gcs bucket.

The data is downloaded for the storm Zeynep (2022-02-18, 10:00, 16:00, 22:00).

Data downloaded for the GraphCast checkpoints GraphCast, GraphCast_small and GraphCast_operational

In [ ]:
# GraphCast_small 22:00

prepare_graphcast_input(
    date="2022-02-18",
    times=["10:00", "16:00", "22:00"],
    levels=13,
    resolution=1.0,
    name="zeynep_22_graphcast_small",
    upload_to_gcs=True
)

In [ ]:
# GraphCast_operational 22:00

prepare_graphcast_input(
    date="2022-02-18",
    times=["10:00", "16:00", "22:00"],
    levels=13,
    resolution=0.25,
    name="zeynep_22_graphcast_operational",
    upload_to_gcs=True
)

In [ ]:
# Download the input data from GCS to the local path
path_gc_small_bucket = "input_data/zeynep_22_graphcast_small/graphcast_ready_input_zeynep_22_graphcast_small.nc"
path_gc_small_local = "graphcast_ready_input_zeynep_22_graphcast_small.nc"

download_file(path_gc_small_bucket, path_gc_small_local)

# Run the GraphCast small model
run_forecast(
    model_name="graphcast_small",
    input_path=path_gc_small_local,
    output_name="zeynep_22_prediction_graphcast_small",
    upload_to_gcs=True
)



In [ ]:
# Download the input data from GCS to the local path
path_gc_operational_bucket = "input_data/zeynep_22_graphcast_operational/graphcast_ready_input_zeynep_22_graphcast_operational.nc"
path_gc_operational_local = "graphcast_ready_input_zeynep_22_graphcast_operational.nc"

download_file(path_gc_operational_bucket, path_gc_operational_local)

# Run the forecast using the downloaded input data
run_forecast(
    model_name="graphcast_operational",
    input_path=path_gc_operational_local,
    output_name="zeynep_22_prediction_graphcast_operational",
    upload_to_gcs=True

# Download Curve Data + Forecast

### Download Real Data

In [ ]:
###### 1.0 resolution ######

# Create CDS API client
c = cdsapi.Client()

# Download ERA5 wind data for February 18, 19, 2022
c.retrieve(
    'reanalysis-era5-single-levels',
    {
        'product_type': 'reanalysis',
        'variable': ['10m_u_component_of_wind', '10m_v_component_of_wind'],
        'year': '2022',
        'month': '02',
        'day': ['18', '19'],
        'time': [f"{h:02d}:00" for h in range(24)],  # hourly
        'format': 'netcdf',
        'area': [53, 8, 55, 10],  # Bounding box around Büsum (N, W, S, E)
        'grid': [0.1, 0.1], 
    },
    'data/north_germany_wind_20220218_19_1res.nc'
)

2025-05-27 16:43:00,304 INFO [2024-09-26T00:00:00] Watch our [Forum](https://forum.ecmwf.int/) for Announcements, news and other discussed topics.
INFO:datapi.legacy_api_client:[2024-09-26T00:00:00] Watch our [Forum](https://forum.ecmwf.int/) for Announcements, news and other discussed topics.
2025-05-27 16:43:00,305 WARNING [2024-06-16T00:00:00] CDS API syntax is changed and some keys or parameter names may have also changed. To avoid requests failing, please use the "Show API request code" tool on the dataset Download Form to check you are using the correct syntax for your API request.
2025-05-27 16:43:01,321 INFO Request ID is bb126d40-5f1b-4b91-ba03-9c83fe0c84b8
INFO:datapi.legacy_api_client:Request ID is bb126d40-5f1b-4b91-ba03-9c83fe0c84b8
2025-05-27 16:43:01,405 INFO status has been updated to accepted
INFO:datapi.legacy_api_client:status has been updated to accepted
2025-05-27 16:43:22,752 INFO status has been updated to running
INFO:datapi.legacy_api_client:status has been upd

af74422ff94f21cf24298da2a3b97620.nc:   0%|          | 0.00/164k [00:00<?, ?B/s]

'data/north_germany_wind_20220218_19_1res.nc'

### Multiple Steps

In [ ]:
# ---------- Download Input Data Multiple Steps ---------- 

def download_every_6h_to_end_of_18th(
    start_date: str,
    start_time: str,
    end_date: str = "2022-02-18",
    step_hours: int = 6,
    levels: int = 13,
    resolution: float = 1.0,
    output_folder: str = "data/prediction_curve/input_curve_7_steps",
    name_prefix: str = "zeynep"
):
    """
    Loop from start_date+start_time every `step_hours` hours until end_date 23:59,
    and for each timestamp call prepare_graphcast_input with n_steps=1.
    """
    # build start/end datetimes
    dt = datetime.strptime(f"{start_date} {start_time}", "%Y-%m-%d %H:%M")
    dt_end = datetime.strptime(f"{end_date} 23:59", "%Y-%m-%d %H:%M")
    
    while dt <= dt_end:
        date_str = dt.strftime("%Y-%m-%d")
        time_str = dt.strftime("%H:%M")
        # tailor the name so each run is unique
        name = f"{name_prefix}_{date_str}_{time_str.replace(':','')}"
        
        print(f"→ downloading for {date_str} @ {time_str}")
        prepare_graphcast_input(
            date          = date_str,
            start_time    = time_str,
            n_steps       = 7,               
            step_hours    = step_hours,      
            levels        = levels,
            resolution    = resolution,
            output_folder = output_folder,
            name          = name,
            upload_to_gcs = False
        )
        dt += timedelta(hours=1)

download_every_6h_to_end_of_18th(
    start_date   = "2022-02-17",
    start_time   = "12:00",
    end_date     = "2022-02-18",
    step_hours   = 6,
    levels       = 13,
    resolution   = 1.0,
    output_folder= "data/prediction_curve/input_curve_7_steps",
    name_prefix  = "zeynep_graphcast"
)


In [ ]:
# ---------- Run Forecast Multiple Steps ----------

# Initialize starting times
time_str = "1200"
start_str = "0000"

# Convert to datetime objects
time = datetime.strptime(time_str, "%H%M")
start = datetime.strptime(start_str, "%H%M")

# Loop until time reaches 17:00
while time <= datetime.strptime("1700", "%H%M"):
    # Format current times
    time_fmt = time.strftime("%H%M")
    start_fmt = start.strftime("%H%M")

    # Run forecast
    run_forecast(
        model_name="graphcast_small",
        input_path=(
            f"data/prediction_curve/input_curve_7_steps/graphcast_ready_input_zeynep_graphcast_2022-02-17_{time_fmt}.nc"
        ),
        output_name=f"curve_prediction_7_steps_2022-02-18_start_{start_fmt}",
        upload_to_gcs=False,
        predictions_folder="data/prediction_curve/predictions_curve_7_steps",
        forecast_hours=30
    )

    # Increment times by 1 hour
    time += timedelta(hours=1)
    start += timedelta(hours=1)



### Single Steps

In [ ]:
# ---------- Download Data Single Steps ----------

def download_every_6h_to_end_of_18th(
    start_date: str,
    start_time: str,
    end_date: str = "2022-02-19",
    step_hours: int = 6,
    levels: int = 13,
    resolution: float = 1.0,
    output_folder: str = "data/prediction_curve/input_curve",
    name_prefix: str = "zeynep"
):
    """
    Loop from start_date+start_time every `step_hours` hours until end_date 23:59,
    and for each timestamp call prepare_graphcast_input with n_steps=1.
    """
    # build start/end datetimes
    dt = datetime.strptime(f"{start_date} {start_time}", "%Y-%m-%d %H:%M")
    dt_end = datetime.strptime(f"{end_date} 23:59", "%Y-%m-%d %H:%M")
    
    while dt <= dt_end:
        date_str = dt.strftime("%Y-%m-%d")
        time_str = dt.strftime("%H:%M")
        # tailor the name so each run is unique
        name = f"{name_prefix}_{date_str}_{time_str.replace(':','')}"
        
        print(f"→ downloading for {date_str} @ {time_str}")
        prepare_graphcast_input(
            date          = date_str,
            start_time    = time_str,
            n_steps       = 3,               # one 6-h slice per call
            step_hours    = step_hours,      # unused when n_steps=1
            levels        = levels,
            resolution    = resolution,
            output_folder = output_folder,
            name          = name,
            upload_to_gcs = False
        )
        dt += timedelta(hours=1)

download_every_6h_to_end_of_18th(
    start_date   = "2022-02-19",
    start_time   = "00:00",
    end_date     = "2022-02-19",
    step_hours   = 6,
    levels       = 13,
    resolution   = 1.0,
    output_folder= "data/prediction_curve/input_curve",
    name_prefix  = "zeynep_graphcast"
)


In [ ]:
# ---------- Run Forecast Single Steps ----------

for t in range(1200, 1800, 100):
    time = f"{t:04d}"   # formats 1300, 1400, …, 2300
    date = "2022-02-19"
    prediction_time = t-1200
    prediction_time = f"{prediction_time:04d}"  # formats 0100, 0200, …, 1200
    print(f"Running forecast for time = {time}…")
    run_forecast(
        model_name="graphcast_small",
        input_path=(
            f"data/prediction_curve/input_curve/"
            f"graphcast_ready_input_zeynep_graphcast_2022-02-18_{time}.nc"
        ),
        output_name=f"curve_prediction_one_step_{date}_{prediction_time}",
        upload_to_gcs=False,
        predictions_folder="data/prediction_curve/predictions_curve_one_step"
    )